# Grant and proof

An issuer grants a bounded operation; the holder proves one request; the resource checks it with the published verifier.

Open this file in JupyterLab or VS Code with the **Deno** kernel. From a repository checkout, install Deno 2.5.6, Python 3.13.9 and uv 0.9.9 (the `.tool-versions` pins), run `uv sync --frozen` and `uv run --frozen deno jupyter --install`, then select Deno and Run All. Any Jupyter-capable editor can use that installed kernel. See [launch and automated checks](README.md).

The saved notebook installs exact published packages. `pnpm notebooks:local` checks the packed candidate; `pnpm notebooks:published` runs these cells unchanged against npm. Fixed times describe a demonstration, not current credentials. Keys and signatures are generated, never golden output. Host authorization, replay and revocation stay outside this notebook.


In [1]:
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
if (signerMetadata.version !== "2.0.0" || verifierMetadata.version !== "2.0.0") throw new Error("package version mismatch");
console.log("exact signer and verifier releases loaded");

exact signer and verifier releases loaded


In [2]:
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import { jwkFromPublicKey, thumbprint, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import type { Es256KeyHandle, KeyHandle, KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";

const fromBase64url = (text: string): Uint8Array => new Uint8Array(Buffer.from(text, "base64url"));

/** An Ed25519 handle over a fresh in-process key (contract-majors 1 and 2). */
function testKeyHandle(keyId: string, role: KeyRole = "holder"): KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const raw = fromBase64url(publicKey.export({ format: "jwk" }).x!);
  return {
    sign: (message) => new Uint8Array(sign(null, message, privateKey)),
    publicKey: () => raw,
    thumbprint: () => thumbprint(jwkFromPublicKey(raw)),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}

/** An ES256 handle over a fresh in-process P-256 key (contract-major 3). */
function testEs256KeyHandle(keyId: string, role: KeyRole = "holder"): Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const raw = new Uint8Array([0x04, ...fromBase64url(x), ...fromBase64url(y)]);
  return {
    sign: (message) => new Uint8Array(sign("sha256", message, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => raw,
    thumbprint: () => v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}
console.log("demonstration key constructors ready");

demonstration key constructors ready


In [3]:
import { signGrant, signReport } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { checkEnvelope, strUtf8, type Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";


const issuer = testKeyHandle("issuer-key-1", "issuer"); // the authorization server's key
const agent = testKeyHandle("agent-key-1");             // the holder: the agent's own key

// 1. The issuer grants the agent one operation on one repository, for two hours.
const grant = await signGrant({
  issuer: "https://issuer.example",
  grantId: "urn:example:grant:1",
  audiences: ["https://code.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_232_800,
  holderThumbprint: await agent.thumbprint(),
  operations: [{
    name: "read_file",
    selectors: [{ kind: "equals", path: ["repository"], value: { t: "string", v: strUtf8("example/website") } }],
  }],
}, issuer);
if (!grant.ok) throw new Error(`signGrant: ${grant.error}`);

// 2. The agent proves one request under that grant.
const args: Tagged = { t: "object", v: new Map<string, Tagged>([
  ["repository", { t: "string", v: strUtf8("example/website") }],
  ["path", { t: "string", v: strUtf8("README.md") }],
]) };
const request = {
  method: "GET",
  targetUri: "https://code.example/files",
  invocationId: "0f8d5a5e-7c3b-4e7a-9a51-2f4c1d6b8e90", // a lowercase RFC 4122 UUID
  operation: "read_file",
  castArguments: args,
};
const envelope = await signReport({ grantCompact: grant.value.grant, ...request }, agent, { issuedAt: 1_767_225_660 });
if (!envelope.ok) throw new Error(`signReport: ${envelope.error}`);

// 3. The resource server verifies the pair with the verifier package and gets facts, not a decision.
const facts = checkEnvelope(envelope.value.grant, envelope.value.proof, {
  ...request,                       // the request the server actually received
  trustedIssuer: { keyId: "issuer-key-1", publicKey: await issuer.publicKey() }, // a key the server already trusts
  issuer: "https://issuer.example", // the issuer the grant must name
  audience: "https://code.example", // this server: one of the grant's audiences
  evaluationTime: 1_767_225_700,    // "now", in Unix seconds: the verifier reads no clock
  clockSkew: 60,                    // seconds of tolerance between the parties' clocks
  proofMaxAge: 300,                 // seconds a proof stays acceptable after its issuedAt
  nonce: { kind: "not_required" },  // this server issues no nonces
});
if (!facts.ok) throw new Error("the envelope did not verify");
console.log(`verified: ${facts.value.operation} under ${facts.value.grantId}`);

verified: read_file under urn:example:grant:1


In [4]:
const refused = checkEnvelope(envelope.value.grant, envelope.value.proof, {
  ...request, method: "PUT", trustedIssuer: { keyId: "issuer-key-1", publicKey: await issuer.publicKey() },
  issuer: "https://issuer.example", audience: "https://code.example", evaluationTime: 1_767_225_700,
  clockSkew: 60, proofMaxAge: 300, nonce: { kind: "not_required" },
});
if (refused.ok) throw new Error("changed request accepted");
console.log("changed request refused; host authorization still required");

changed request refused; host authorization still required


## Cleanup

These cells keep keys and compacts only in the kernel. Shut down the kernel to discard them. No private material is persisted. The runner shuts down the kernel and removes its consumer directory, package caches and temporary kernel registration, and fails on files left in the notebook working directory.

Continue with the [guides](../guides/getting-started.md).
